In [ ]:
# Import libraries

import pandas as pd, sqlalchemy as sqla, decouple
from sqlalchemy import Integer, String, Numeric
from IPython.display import display

clone_engine = sqla.create_engine(decouple.config("MIS_DB"))
dashboard_engine = sqla.create_engine(decouple.config("MIS_DASHBOARD"))
local_engine = sqla.create_engine(decouple.config("MIS_DB_LOCAL"))
mis_file_path = decouple.config("MIS_FILE_PATH")

In [ ]:
# Import RTV data from Excel

rtv_data = pd.read_excel(rf"{mis_file_path}\Report Templates\Daily Sales\RTV Data.xlsx")
rtv_data = rtv_data.rename(columns={"product_class":"class", "product_type":"type"})
display(rtv_data.columns)

In [ ]:
# Select columns from raw rtv_data and filter rows

clean_rtv_data = rtv_data[['year', 'month', 'sales_group', 'account_name', 'account_chain', 'account_type', 'lead_name', 'product_code', 'product_name', 'class', 'brand', 'type', 'net_amount',]]
clean_rtv_data["net_amount"] = clean_rtv_data["net_amount"].abs()

print("clean_rtv_data net_amount: ", clean_rtv_data["net_amount"].sum())

# included_months = ["January", "February", "March", "April", "May", "June"]

rtv_data_2026 = clean_rtv_data[clean_rtv_data["year"] == 2026]
# clean_rtv_data = clean_rtv_data[clean_rtv_data["month"].isin(included_months)]
# display(rtv_data_2026.sum())
print("distinct years: ", clean_rtv_data["year"].unique())
print("distinct months: ", clean_rtv_data["month"].unique())
print("Filtered net_amount: ", clean_rtv_data["net_amount"].sum())

In [ ]:
# Merge ref.product_details and rtv dataframes 

ref_product_details = pd.read_sql("SELECT * FROM ref.product_details", clone_engine)
final_rtv_data = pd.merge(clean_rtv_data, ref_product_details, how="left", left_on="product_code", right_on="product_code1")
# print(final_rtv_data.columns)

columns_to_drop = [column for column in final_rtv_data.columns if column.endswith('_x')]
final_rtv_data = final_rtv_data.drop(columns=columns_to_drop)
final_rtv_data = final_rtv_data.rename(columns=lambda x: x.replace("_y", ""))
print("Renamed columns: \n", final_rtv_data.columns,"\n")
final_rtv_data = final_rtv_data[['year', 'month', 'sales_group', 'account_name', 'account_chain', 'account_type', 'lead_name', 'product_code', 'main_code', 'product_name', 'class', 'brand', 'type', 'net_amount']]
print("Final columns: \n", final_rtv_data.columns, "\n")

null_rtv = final_rtv_data[ final_rtv_data[['year', 'month', 'sales_group', 'account_name', 'account_chain', 'account_type', 'lead_name', 'product_code', 'main_code', 'product_name', 'class', 'brand', 'type', 'net_amount']].isna().any(axis=1) ]
display(null_rtv)

In [ ]:
# Anti-join

rtv_mtd = final_rtv_data.copy()

try:
    existing = pd.read_sql("SELECT * FROM `dashboard-sales`.rtv", con=dashboard_engine)
    existing = existing.drop(columns=["id"], errors="ignore")

    for df in [rtv_mtd, existing]:
        df["year"] = df["year"].astype("int64")
        df["net_amount"] = pd.to_numeric(df["net_amount"], errors="coerce").round(5)

    key_cols = list(rtv_mtd.columns)
    rtv_mtd["__occ"] = rtv_mtd.groupby(key_cols).cumcount()
    existing["__occ"] = existing.groupby(key_cols).cumcount()

    merged = rtv_mtd.merge(existing, on=key_cols + ["__occ"], how="left", indicator=True)
    new_rows = merged[merged["_merge"] == "left_only"].drop(columns=["_merge", "__occ"])
    rtv_mtd = rtv_mtd.drop(columns=["__occ"])

except Exception:
    new_rows = rtv_mtd

print(f"New rows to insert: {len(new_rows)}")
new_rows.shape

In [ ]:
# Export data to database
final_rtv_data.loc[final_rtv_data["account_chain"] == "E-Commerce", "lead_name"] = "ECOM"
# final_rtv_data = final_rtv_data[final_rtv_data["account_chain"] == "E-Commerce"]
# final_rtv_data.head()
    
final_rtv_data.to_sql(
    name="rtv",
    con=dashboard_engine,
    schema="dashboard-sales",
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

In [ ]:
# Enforce NOT NULL with no default on every sell_out_staging column
# (dtype= on to_sql only shapes CREATE TABLE; it doesn't alter an existing table)

# alter_sql = """
# ALTER TABLE `dashboard-sales`.`rtv`
#     MODIFY `year`          INT           NOT NULL,
#     MODIFY `month`         VARCHAR(20)   NOT NULL,
#     MODIFY `sales_group`   VARCHAR(50)   NOT NULL,
#     MODIFY `account_name`  VARCHAR(255)  NOT NULL,
#     MODIFY `account_chain` VARCHAR(255)  NOT NULL,
#     MODIFY `account_type`  VARCHAR(100)  NOT NULL,
#     MODIFY `lead_name`     VARCHAR(255)  ,
#     MODIFY `product_code`  VARCHAR(50)   NOT NULL,
#     MODIFY `main_code`     VARCHAR(255)  NOT NULL,
#     MODIFY `product_name`  VARCHAR(255)  NOT NULL,
#     MODIFY `class`         VARCHAR(100)  NOT NULL,
#     MODIFY `brand`         VARCHAR(100)  NOT NULL,
#     MODIFY `type`          VARCHAR(100)  NOT NULL,
#     MODIFY `net_amount`    DECIMAL(15,5) UNSIGNED NOT NULL
# """

# with dashboard_engine.begin() as conn:
#     conn.execute(sqla.text(alter_sql))

In [ ]:
rtv_dashboard = pd.read_sql("SELECT * FROM `dashboard-sales`.rtv", dashboard_engine)
rtv_dashboard.head()